# Ovarian cancer data analysis
Train steamboat model on HGSC data.

In [2]:
import commot as ct
import scanpy as sc
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import scipy as sp

In [3]:
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

In [4]:
# https://www.nature.com/articles/s41590-024-01943-5

## Load data

In [5]:
adata = sc.read_h5ad("G:/data/slidetags/HumanTonsil.h5ad")

In [6]:
df_ligrec=ct.pp.ligand_receptor_database(database='CellChat', species='human')

In [7]:
ct.tl.spatial_communication(adata,
        database_name='user_database', df_ligrec=df_ligrec, dis_thr=200, heteromeric=True)
adata.write_h5ad(f"saved_results/tonsil_commot.h5ad")

In [8]:
adata = sc.read_h5ad(f"saved_results/tonsil_commot.h5ad")

cell_type_key = 'cluster'
pseudocount = 20.
temp = pd.DataFrame(0,
                    index=adata.obs[cell_type_key].cat.categories, 
                    columns=adata.obs[cell_type_key].cat.categories)

for i in temp.index:
    mask_i = adata.obs[cell_type_key] == i
    for j in temp.columns:
        mask_j = adata.obs[cell_type_key] == j
        sub_attnp = adata.obsp['commot-user_database-total-total'][mask_i, :][:, mask_j]
        normalization_factor = sub_attnp.nnz + pseudocount
        temp.loc[i, j] = sub_attnp.sum() / normalization_factor
temp.to_csv(f"saved_results/tonsil_commot_cci.csv")
commot_cci = temp